In [10]:
from cr_element import CRElement
from cr_gamestate import CRGameState
from cr_dataset import CRDataset
import matplotlib.pyplot as plt
import cv2
import os
import json
import shutil
import pyarrow as pa
import pyarrow.parquet as pq
from PIL import Image
from io import BytesIO
import numpy as np


cr_dataset = CRDataset()

In [ ]:
# h0, w0 = replay[0].shape[:2]
# writer = cv2.VideoWriter("annotated13.mp4", 0, 10, (w0, h0), isColor=True)
def get_event_log(replay, deck):
    old_hand = [None, None, None, None]
    prev_elixir = 0
    queued_elixir_losses = 0
    # played = []
    last_true_update_frame = 0
    event_log = []

    for frame_num, frame in enumerate(replay):
        elixir = CRGameState.current_elixir(frame, cr_dataset._elixirs)
        if elixir is not None:
            if(elixir - prev_elixir < -0.8): queued_elixir_losses += 1
            prev_elixir = elixir

        my_hand = CRGameState.cards_in_hand(frame, deck, elixir)
        empties = [index for index, element in enumerate(my_hand) if element == "empty"]
        nones = [index for index, element in enumerate(my_hand) if element is None]
        new_played = 0
        if(len(empties) > 0):
            for empty_index in empties:
                if old_hand[empty_index] != "empty":
                    # played.append(old_hand[empty_index])
                    event_log.append((last_true_update_frame, old_hand[empty_index]))
                    new_played += 1
                    old_hand[empty_index] = "empty"
                    # print(empty_index)
        
        
        for i, card in enumerate(my_hand):
            if len(nones) > 0 or len(empties) > 0:
                continue
            # gray = "gray" in card
            # gray_old = old_hand[i] is not None and "gray" in old_hand[i]
            name = card
            if(old_hand[i] != name):
                if(queued_elixir_losses > 0 and old_hand[i] is not None and "gray_" + name != old_hand[i] and "gray_" + old_hand[i] != name):

                    # played.append(old_hand[i])
                    event_log.append((last_true_update_frame, old_hand[i]))
                    new_played += 1
                    # print(old_hand, my_hand)
                    # print(f"Played {old_hand[i]} replaced by {name}")
                old_hand[i] = name
            # cv2.putText(frame, name, (CRElement.card.origin[0] + i * (CRElement.card_space + CRElement.card.size[0]), CRElement.card.origin[1]- i * 8), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,0) if not gray else (255, 255, 255), 2)
        # TODO if empty appears, there is likely occlusion. Report card that empty replaced as played, but wait to update old hand.
        queued_elixir_losses -= new_played
        if (queued_elixir_losses < 0): queued_elixir_losses = 0
        if(len(empties) + len(nones) == 0): last_true_update_frame = frame_num
        # cv2.putText(frame, str(queued_elixir_losses), (40, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 5)
        # for i, card in enumerate(played):
        #     if(card == "empty"): continue
        #     cv2.putText(frame, card, (0, (i + 1) * 30), cv2.FONT_HERSHEY_SIMPLEX, 2, (255,0,255), 2)

        # If we lost 1 card and elixir went down that much (or that much - 1 if played before elixir addition (how to deal with elixir generators?)),
        # Count it as played
        # Update old_hand
        
        # cv2.putText(frame, str(elixir), (CRElement.elixir.origin[0], CRElement.elixir.origin[1]), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        # cv2.putText(frame, str(frame_num), (CRElement.elixir.origin[0], CRElement.elixir.origin[1]+20), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,0), 2)
        # writer.write(frame)

    # writer.release()
    return event_log
    

In [ ]:
validated_replays = [item['replay'] for item in cr_dataset.manifest]

for replay_name in validated_replays:
    metadata = cr_dataset.get_replay_from_manifest(replay_name)
    if 'card_playtime' in metadata.keys() and metadata['card_playtime'] == True:
        continue
    print(replay_name)
    replay, deck, known_cards = cr_dataset.load_replay(replay_name=replay_name, clear_huggingface=False)
    shutil.rmtree("C:\\Users\\0dps1\\.cache\\huggingface\\hub\\datasets--chrisrca--clash-royale-tv-replays\\blobs")
    shutil.rmtree("C:\\Users\\0dps1\\.cache\\huggingface\\hub\\datasets--chrisrca--clash-royale-tv-replays\\snapshots")
    # h0, w0 = replay[0].shape[:2]
    # writer = cv2.VideoWriter(f"vid.mp4", 0, 10, (w0, h0), isColor=True)
    # for frame in replay:
    #     writer.write(frame)
    # writer.release()
    if(known_cards):
        cr_dataset.get_replay_from_manifest(replay_name)['cards_identified'] = True
        event_log = get_event_log(replay, deck)
        for frame, name in event_log:
            os.makedirs(f"replays/{replay_name}", exist_ok=True)
            cv2.imwrite(f"replays/{replay_name}/{name}-{frame}.png", replay[frame])
            cv2.imwrite(f"replays/{replay_name}/{name}-{frame}-next.png", replay[frame + 1])
        cr_dataset.get_replay_from_manifest(replay_name)['card_playtime'] = True
        with open("manifest.json", "w") as json_file:
            json.dump(cr_dataset.manifest, json_file, indent=4)
    else:
        cr_dataset.get_replay_from_manifest(replay_name)['cards_identified'] = False
        print(f"Replay {replay_name} may have a new card")
        with open("manifest.json", "w") as json_file:
            json.dump(cr_dataset.manifest, json_file, indent=4)
    # Check if replay in manifest and cards_identified
    # if len([item for item in cr_dataset.manifest if item['replay'] == replay_name]) > 0:
    #     continue

    # replay = cr_dataset.load_replay(replay_name=replay_name)

In [13]:
def create_pyarrow_table():
    arenas = os.listdir("replays")
    data = {
        "card": [],
        "png_bytes": [],
        "x": [],
        "y": [],
        "arena": [],
        "replay": [],
        "frame": [] 
    }
    i = 0
    for arena in arenas:
        replays = os.listdir(f"replays/{arena}")
        for replay in replays:
            images = os.listdir(f"replays/{arena}/{replay}")
            for image_path in images:
                image = cv2.imread(f"replays/{arena}/{replay}/{image_path}")
                has_card_been_played = "next" in image_path
                gray = "gray" in image_path
                if(not has_card_been_played or gray): continue
                image = CRElement.cut_to_fit(image, CRElement.arena)
                is_success, buffer = cv2.imencode(".png", image)
                if is_success:
                    data["card"].append(image_path.split("-")[0])
                    data["png_bytes"].append(buffer.tobytes())
                    data["x"].append(-1)
                    data["y"].append(-1)
                    data["arena"].append(arena)
                    data["replay"].append(replay)
                    data["frame"].append(int(image_path.split(".")[0].split("-")[1]))
                    i += 1
                    if i % 100 == 0:
                        print(i)
    return pa.table({
        "card": pa.array(data["card"], type=pa.string()),
        "png_bytes": pa.array(data["png_bytes"], type=pa.binary()),
        "x": pa.array(data["x"], type=pa.int16()),
        "y": pa.array(data["y"], type=pa.int16()),
        "arena": pa.array(data["arena"], type=pa.string()),
        "replay": pa.array(data["replay"], type=pa.string()),
        "frame": pa.array(data["frame"], type=pa.int16()),
    })

# table = pa.Table.from_pylist(frames_buffer)
# parquet_path = replay_dir / "frames.parquet"
# pq.write_table(
#     table,
#     str(parquet_path),
#     compression="zstd",
#     use_dictionary=True,
#     write_statistics=True
# )
                

table = create_pyarrow_table()
parquet_path = "placement.parquet"
pq.write_table(
    table,
    str(parquet_path),
    compression="zstd",
    use_dictionary=True,
    write_statistics=True
)

100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000


In [14]:
parquet = pq.read_table(parquet_path)
column = parquet["png_bytes"]
png_byte_list = column.to_pylist()
dataset = [cv2.cvtColor(np.array(Image.open(BytesIO(img))), cv2.COLOR_RGB2BGR) for img in png_byte_list]
cv2.imshow("test", dataset[0])
cv2.waitKey(0)

-1

In [ ]:
clock = cv2.imread("clock.png")
gray = cv2.cvtColor(clock, cv2.COLOR_BGR2GRAY)
mask = (gray > 5).astype(np.uint8) * 255
screenshot = dataset[21]
res = cv2.matchTemplate(screenshot, clock, cv2.TM_CCORR_NORMED, mask=mask)
_, max_val, _, match_point = cv2.minMaxLoc(res)
match_point = (match_point[0] + 34/2, match_point[1] + 40/2)
print(max_val)
print(match_point)
# Assume clock and dataset[18] are already loaded
h, w = clock.shape[:2]  # height and width of template

# Top-left corner from minMaxLoc
top_left = (int(match_point[0] - w/2), int(match_point[1] - h/2))
bottom_right = (top_left[0] + w, top_left[1] + h)

# Draw rectangle on a copy of the image
img_copy = screenshot.copy()
cv2.rectangle(img_copy, top_left, bottom_right, color=(0, 255, 0), thickness=2)

cv2.imshow("Matched Area", img_copy)
cv2.waitKey(0)
cv2.destroyAllWindows()

0.5371074676513672
(252.0, 518.0)


In [ ]:
# arrows = cv2.imread("potential_new_cards\\b613634d-24f8-4e53-9feb-aef51e5c4a99.png")
# print(CRGameState.is_card_sliding(arrows))

# Incorrect replays
# replays\arena_31\f7db1ae0-7e87-4dd4-991a-d88ff88bf262\gray_minions-1139.png
# replays\arena_31\efb06ad5-5a2a-45f6-8e72-6d557d1654d3\gray_mini_pekka-699-next.png
# replays\arena_31\efb06ad5-5a2a-45f6-8e72-6d557d1654d3\gray_golem-607-next.png
# replays\arena_31\e73c9c99-867d-45fe-b072-d79961965947\gray_fireball-565.png
# replays\arena_31\e69bbe32-8712-475d-80fe-df783430726b\gray_zap-639.png
# replays\arena_31\e73c9c99-867d-45fe-b072-d79961965947\gray_hog_rider-379.png

In [ ]:
# arrows = cv2.imread("potential_new_cards\\1dfcde88-9396-4fd3-9751-bf09159005c0.png")
# arrows_real = cv2.imread("cards\\evo_mega_knight-7.png")
# best_score, best_match = CRElement.best_match(
#     template=CRElement.prep(arrows), # TODO test different prepping downscaling for gray & non-gray (1,4)
#     labelled_image_collection={"arrows": CRElement.prep(arrows_real)},
#     shearing=(9,3)
# )
# print(best_score)